# 실습 Agent 과제

## 예시

1. 사용자 질문에서 키워드를 뽑아서
2. 네이버 뉴스 검색 API를 활용해 뉴스를 받아오고
3. 해당 뉴스들을 요약하여 사용자에게 전달하는 Agent
4. 서비스 주제를 하나 정해서, AI와 상의하면서 해보기.

In [3]:
import datetime
import json
import os
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

DATA_FILE = "todos.json"

# 로컬 JSON 파일 읽기/쓰기 헬퍼 함수
def _load_todos() -> list:
    if not os.path.exists(DATA_FILE):
        return []
    with open(DATA_FILE, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            return []

def _save_todos(todos: list):
    with open(DATA_FILE, "w", encoding="utf-8") as f:
        json.dump(todos, f, ensure_ascii=False, indent=2)


# 1. Tool 정의
def add_todo(task: str, due_date: str = "기한 없음") -> str:
    """새로운 할 일(일정)을 목록에 추가합니다.
    
    Args:
        task: 등록할 할 일 내용 (str)
        due_date: 마감일 또는 시간 (str, 기본값 '기한 없음')
    """
    todos = _load_todos()
    new_id = len(todos) + 1 if not todos else max(t["id"] for t in todos) + 1
    
    new_item = {
        "id": new_id,
        "task": task,
        "due_date": due_date,
        "status": "진행중"
    }
    todos.append(new_item)
    _save_todos(todos)
    return f"할 일이 성공적으로 추가되었습니다: [ID: {new_id}] {task} (기한: {due_date})"


def list_todos() -> str:
    """현재 등록된 모든 할 일과 완료 상태 목록을 조회합니다."""
    todos = _load_todos()
    if not todos:
        return "현재 등록된 할 일이 없습니다."
    return json.dumps(todos, ensure_ascii=False, indent=2)


def complete_todo(task_id: int) -> str:
    """특정 ID를 가진 할 일을 '완료' 상태로 변경합니다.
    
    Args:
        task_id: 완료 처리할 할 일의 번호 ID (int)
    """
    todos = _load_todos()
    for item in todos:
        if item["id"] == task_id:
            item["status"] = "완료"
            _save_todos(todos)
            return f"ID {task_id}번 '{item['task']}' 작업이 완료 처리되었습니다."
    return f"ID {task_id}번에 해당하는 할 일을 찾을 수 없습니다."


def delete_todo(task_id: int) -> str:
    """특정 ID를 가진 할 일을 목록에서 완전히 삭제합니다.
    
    Args:
        task_id: 삭제할 할 일의 번호 ID (int)
    """
    todos = _load_todos()
    filtered_todos = [item for item in todos if item["id"] != task_id]
    
    if len(todos) == len(filtered_todos):
        return f"ID {task_id}번에 해당하는 할 일을 찾을 수 없습니다."
    
    _save_todos(filtered_todos)
    return f"ID {task_id}번 할 일이 성공적으로 삭제되었습니다."


# 2. System Prompt 구성
now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

system_prompt = f"""당신은 사용자의 일상 업무와 할 일을 체계적으로 관리해주는 개인 일정 관리 비서입니다.

[기본 정보]
- 현재 일시: {now}

[행동 지침]
1. 사용자가 할 일을 추가해달라고 하면 `add_todo` 도구를 실행하세요.
2. 현재 일정이나 목록을 확인하고자 하면 `list_todos` 도구를 실행해 목록을 확인하고 정리해서 전달하세요.
3. 특정 작업을 마쳤다고 하면 목록을 확인해 해당 ID를 찾은 후 `complete_todo` 도구로 완료 처리하세요.
4. 특정 일정이나 할 일을 삭제/취소해달라고 하면 `delete_todo` 도구를 실행하여 삭제하세요.
5. ID가 불명확할 때는 먼저 `list_todos`로 목록을 조회한 후 맞는 항목의 ID를 대상으로 작업하세요.
6. 한국어로 정중하고 간결하게 답변하세요.
"""

# 3. Memory & Agent 생성
memory = InMemorySaver()

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[add_todo, list_todos, complete_todo, delete_todo],
    system_prompt=system_prompt,
    checkpointer=memory,
)

# 4. 세션 설정 및 실행 루프
thread_config = {
    "configurable": {"thread_id": "todo_session_1"}
}

print("=" * 60)
print("📝 개인 일정 관리 비서가 시작되었습니다. (종료: '종료')")
print("테스트 명령:")
print(" - 등록: '내일 오후 2시 팀 미팅 일정 등록해줘'")
print(" - 조회: '할 일 목록 보여줘'")
print(" - 완료: '팀 미팅 완료 처리해줘'")
print(" - 삭제: '팀 미팅 일정 삭제해줘'")
print("=" * 60)

user_input = input()

while user_input != "종료":
    print("사용자:", user_input)
    result = agent.invoke(
        {"messages": [{"role": "user", "content": user_input}]},
        thread_config,
    )
    print("AI:", result["messages"][-1].content)
    user_input = input()

📝 개인 일정 관리 비서가 시작되었습니다. (종료: '종료')
테스트 명령:
 - 등록: '내일 오후 2시 팀 미팅 일정 등록해줘'
 - 조회: '할 일 목록 보여줘'
 - 완료: '팀 미팅 완료 처리해줘'
 - 삭제: '팀 미팅 일정 삭제해줘'
사용자: 일정 확인해줘
AI: 현재 등록된 일정은 다음과 같습니다.
1. 추가과제 완료하기 - 마감일 2026-08-31 18:00
2. 등교 - 마감일 2026-09-01 09:00
3. 9월 수업일지 새로 만들기 - 마감일 2026-09-01 11:30

도움 필요하신 일정에 대해 말씀해 주세요.
사용자: 2번 등교하기 삭제해줘
AI: 2번 '등교' 일정이 성공적으로 삭제되었습니다. 다른 일정에 대해 도와드릴까요?
사용자: 내일 점심시간 2시까지 8월 수업일지 정리완료하기 추가해줘
AI: 내일 점심시간 2시까지 '8월 수업일지 정리완료하기' 일정이 추가되었습니다. 다른 일정도 관리해 드릴까요?
사용자: 8월 수업일지 정리완료하기는 오후5시까지로 수정해줘
AI: '8월 수업일지 정리완료하기' 일정이 오후 5시까지 마감으로 수정되었습니다. 추가로 도와드릴 일이 있을까요?
사용자: 할 일 목록 보여줘
AI: 현재 할 일 목록입니다.
1. 추가과제 완료하기 - 마감일 2026-08-31 18:00
3. 9월 수업일지 새로 만들기 - 마감일 2026-09-01 11:30
4. 8월 수업일지 정리완료하기 - 마감일 2026-09-01 17:00

더 필요하신 것이 있으면 말씀해 주세요.
사용자: 1번 추가과제 완료하기는 완료처리해줘
AI: 1번 '추가과제 완료하기' 작업이 완료 처리되었습니다. 다른 일정도 도와드릴까요?
